Step 1 — Import the libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# make plots and tables a bit nicer
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("All libraries imported successfully ✅")

pandas : 3.0.3
numpy  : 2.4.6
All libraries imported successfully ✅


Step 2 — Load the three CSV files

In [2]:
DATA_DIR = "ashrae-energy-prediction" 

# 1) building metadata — one row per building (site, use, size...)
buildings = pd.read_csv(f"{DATA_DIR}/building_metadata.csv")

# 2) weather — hourly weather per site
weather = pd.read_csv(f"{DATA_DIR}/weather_train.csv", parse_dates=["timestamp"])

# 3) meter readings — the BIG one (~20 million rows)
meters = pd.read_csv(
    f"{DATA_DIR}/train.csv",
    parse_dates=["timestamp"],
    dtype={"building_id": "int16", "meter": "int8", "meter_reading": "float32"},
)

print("buildings :", buildings.shape)   # (rows, columns)
print("weather   :", weather.shape)
print("meters    :", meters.shape)

buildings : (1449, 6)
weather   : (139773, 9)
meters    : (20216100, 4)


Step 3 — Look at the data & understand it

In [3]:
print("===== BUILDINGS =====")
display(buildings.head())

print("===== WEATHER =====")
display(weather.head())

print("===== METERS =====")
display(meters.head())

# how many readings of each meter type?
print("\nMeter type counts:")
print(meters["meter"].value_counts())

===== BUILDINGS =====


,site_id,building_id,primary_use,square_feet,year_built,floor_count
0,0,0,Education,7432,2008.0,NaN
1,0,1,Education,2720,2004.0,NaN
2,0,2,Education,5376,1991.0,NaN
3,0,3,Education,23685,2002.0,NaN
4,0,4,Education,116607,1975.0,NaN


===== WEATHER =====


,site_id,timestamp,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2016-01-01 00:00:00,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
1,0,2016-01-01 01:00:00,24.4,NaN,21.1,-1.0,1020.2,70.0,1.5
2,0,2016-01-01 02:00:00,22.8,2.0,21.1,0.0,1020.2,0.0,0.0
3,0,2016-01-01 03:00:00,21.1,2.0,20.6,0.0,1020.1,0.0,0.0
4,0,2016-01-01 04:00:00,20.0,2.0,20.0,-1.0,1020.0,250.0,2.6


===== METERS =====


,building_id,meter,timestamp,meter_reading
0,0,0,2016-01-01,0.0
1,1,0,2016-01-01,0.0
2,2,0,2016-01-01,0.0
3,3,0,2016-01-01,0.0
4,4,0,2016-01-01,0.0



Meter type counts:
meter
0    12060910
1     4182440
2     2708713
3     1264037
Name: count, dtype: int64


Step 4 — Keep only electricity

In [4]:
# Step 4: keep only electricity readings (meter code 0)
elec = meters[meters["meter"] == 0].copy()

print("all meters   :", meters.shape[0], "rows")
print("electricity  :", elec.shape[0], "rows")
print("buildings with electricity:", elec["building_id"].nunique())
elec.head()

all meters   : 20216100 rows
electricity  : 12060910 rows
buildings with electricity: 1413


,building_id,meter,timestamp,meter_reading
0,0,0,2016-01-01,0.0
1,1,0,2016-01-01,0.0
2,2,0,2016-01-01,0.0
3,3,0,2016-01-01,0.0
4,4,0,2016-01-01,0.0


Step 5 — Merge the three tables into one

In [5]:
# Step 5: merge electricity + building metadata + weather into one table

# a) attach building info (adds site_id, primary_use, square_feet, ...)
df = elec.merge(buildings, on="building_id", how="left")

# b) attach weather, matched on BOTH site and timestamp
df = df.merge(weather, on=["site_id", "timestamp"], how="left")

print("merged shape:", df.shape)
print("\ncolumns now:\n", list(df.columns))
df.head()

merged shape: (12060910, 16)

columns now:
 ['building_id', 'meter', 'timestamp', 'meter_reading', 'site_id', 'primary_use', 'square_feet', 'year_built', 'floor_count', 'air_temperature', 'cloud_coverage', 'dew_temperature', 'precip_depth_1_hr', 'sea_level_pressure', 'wind_direction', 'wind_speed']


,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,0,2016-01-01,0.0,0,Education,7432,2008.0,NaN,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
1,1,0,2016-01-01,0.0,0,Education,2720,2004.0,NaN,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
2,2,0,2016-01-01,0.0,0,Education,5376,1991.0,NaN,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
3,3,0,2016-01-01,0.0,0,Education,23685,2002.0,NaN,25.0,6.0,20.0,NaN,1019.7,0.0,0.0
4,4,0,2016-01-01,0.0,0,Education,116607,1975.0,NaN,25.0,6.0,20.0,NaN,1019.7,0.0,0.0


Step 6: part a — Fix the site-0 units (kBTU → kWh)

In [6]:
# Step 6: part a - fix site-0 electricity units (kBTU -> kWh)
KBTU_TO_KWH = 0.2931   # 1 kWh = 3.412 kBTU, so kBTU * 0.2931 = kWh

# look at site 0 BEFORE the fix
before = df.loc[df["site_id"] == 0, "meter_reading"].max()

# apply the conversion ONLY to site 0 electricity rows
mask = df["site_id"] == 0
df.loc[mask, "meter_reading"] = df.loc[mask, "meter_reading"] * KBTU_TO_KWH

after = df.loc[df["site_id"] == 0, "meter_reading"].max()
print(f"site-0 rows corrected : {mask.sum():,}")
print(f"site-0 max reading  before: {before:,.1f}")
print(f"site-0 max reading  after : {after:,.1f}   (should be ~3.4x smaller)")

site-0 rows corrected : 908,409
site-0 max reading  before: 4,521.0
site-0 max reading  after : 1,325.1   (should be ~3.4x smaller)


Step 6: part b — Remove dead-meter runs 

In [7]:
# Step 6: part b - drop long consecutive-zero streaks (dead meters), keep isolated zeros
df = df.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

is_zero = df["meter_reading"].eq(0)

# give each consecutive same-status streak (within a building) a run id
new_run = is_zero != is_zero.groupby(df["building_id"]).shift()
run_id = new_run.cumsum()

# length of each streak
run_len = df.groupby(run_id)["meter_reading"].transform("size")

# drop rows that are zero AND part of a streak >= 48 hours (2 days)
MIN_DEAD_RUN = 48
drop_mask = is_zero & (run_len >= MIN_DEAD_RUN)

before = len(df)
df = df.loc[~drop_mask].reset_index(drop=True)
print(f"dead-meter rows dropped : {drop_mask.sum():,}")
print(f"rows: {before:,} -> {len(df):,}")

dead-meter rows dropped : 459,015
rows: 12,060,910 -> 11,601,895


Step 6: part c — Tame outlier spikes

In [8]:
# Step 6: part c - cap extreme spikes per building at its 99.9th percentile
caps = df.groupby("building_id")["meter_reading"].transform(lambda s: s.quantile(0.999))

n_clipped = (df["meter_reading"] > caps).sum()
df["meter_reading"] = np.minimum(df["meter_reading"], caps).astype("float32")

print(f"readings clipped down : {n_clipped:,}")
print(f"max reading now        : {df['meter_reading'].max():,.1f}")

readings clipped down : 11,130
max reading now        : 19,511.0


Step 6: part d — Fill the weather gaps

In [9]:
# Step 6: part d - fill missing weather values
weather_cols = ["air_temperature", "cloud_coverage", "dew_temperature",
                "precip_depth_1_hr", "sea_level_pressure",
                "wind_direction", "wind_speed"]

# how much is missing BEFORE?
print("missing weather values BEFORE:")
print(df[weather_cols].isna().sum())

# 1) interpolate along each building's time order (fills interior gaps smoothly)
df = df.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
df[weather_cols] = (
    df.groupby("building_id")[weather_cols]
      .transform(lambda g: g.interpolate(limit_direction="both"))
)

# 2) fill any still-missing (e.g. a column empty for a whole building) with the global median
df[weather_cols] = df[weather_cols].fillna(df[weather_cols].median())

print("\nmissing weather values AFTER:")
print(df[weather_cols].isna().sum())

missing weather values BEFORE:
air_temperature         46341
cloud_coverage        5115458
dew_temperature         48058
precip_depth_1_hr     2490904
sea_level_pressure    1007097
wind_direction         656829
wind_speed              65729
dtype: int64

missing weather values AFTER:
air_temperature       0
cloud_coverage        0
dew_temperature       0
precip_depth_1_hr     0
sea_level_pressure    0
wind_direction        0
wind_speed            0
dtype: int64


Step 7 — Final check & save the finalized dataset

In [11]:
# Step 7: final sanity check + save

# --- sanity checks ---
print("final shape        :", df.shape)
print("date range         :", df['timestamp'].min(), "->", df['timestamp'].max())
print("buildings          :", df['building_id'].nunique())
print("any NaN in reading? :", df['meter_reading'].isna().any())
print("\nmeter_reading summary (kWh):")
print(df['meter_reading'].describe())

# --- save to a fast binary file (Parquet) ---
OUT = "electricity_clean.parquet"
df.to_parquet(OUT, index=False)
df.to_csv("electricity_clean.csv", index=False)
print(f"\n✅ saved finalized dataset -> {OUT}")

final shape        : (11601895, 16)
date range         : 2016-01-01 00:00:00 -> 2016-12-31 23:00:00
buildings          : 1413
any NaN in reading? : False

meter_reading summary (kWh):
count    1.160190e+07
mean     1.649790e+02
std      3.722443e+02
min      0.000000e+00
25%      2.201000e+01
50%      6.395000e+01
75%      1.625000e+02
max      1.951100e+04
Name: meter_reading, dtype: float64

✅ saved finalized dataset -> electricity_clean.parquet
